In [0]:
# ==========================================
# GOLD LAYER - LEAGUE TABLE
# ==========================================

# Read the three Silver datasets
standings_df = spark.table(
    "workspace.default.silver_standings"
)

teams_df = spark.table(
    "workspace.default.silver_teams"
)

competition_df = spark.table(
    "workspace.default.silver_competition"
)

# Inspect record counts
print("Standings:", standings_df.count())
print("Teams:", teams_df.count())
print("Competition:", competition_df.count())

# Inspect available columns
standings_df.printSchema()
teams_df.printSchema()
competition_df.printSchema()


In [0]:

from pyspark.sql.functions import col, round, when

# Filter to overall league standings
total_standings_df = standings_df.filter(
    col("standing_type") == "TOTAL"
)

# Join standings with teams
joined_df = total_standings_df.alias("s").join(
    teams_df.alias("t"),
    col("s.team_id") == col("t.team_id"),
    "left"
)

# Join the result with competition
joined_df = joined_df.join(
    competition_df.alias("c"),
    col("s.season_id") == col("c.season_id"),
    "inner"
)

print("Rows after joining:", joined_df.count())


In [0]:
# ==========================================
# 4. BUILD GOLD LEAGUE TABLE
# ==========================================

gold_df = joined_df.select(

    # Competition information
    col("c.competition_name"),
    col("c.season_id"),
    col("c.season_start"),
    col("c.season_end"),

    # Team information
    col("s.team_id"),
    col("s.team_name"),
    col("t.stadium"),

    # League standings
    col("s.position"),
    col("s.played"),
    col("s.won"),
    col("s.drawn"),
    col("s.lost"),
    col("s.goals_for"),
    col("s.goals_against"),
    col("s.goal_difference"),
    col("s.points"),
    col("s.form"),

    # Calculated performance metrics
    when(
        col("s.played") > 0,
        round(col("s.points") / col("s.played"), 2)
    ).alias("points_per_game"),

    when(
        col("s.played") > 0,
        round(col("s.goals_for") / col("s.played"), 2)
    ).alias("goals_per_game")
)

display(gold_df.orderBy("position"))


In [0]:
# ==========================================
# 5. GOLD DATA QUALITY CHECKS
# ==========================================

total_rows = gold_df.count()

unique_teams = gold_df.select(
    "season_id", "team_id"
).distinct().count()

# Ensure we haven't lost or duplicated records
assert total_rows == total_standings_df.count(), \
    "Row count changed during joins!"

assert total_rows == unique_teams, \
    "Duplicate team-season records detected!"

# Ensure every team has a stadium
missing_stadiums = gold_df.filter(
    col("stadium").isNull()
).count()

assert missing_stadiums == 0, \
    "Some teams are missing stadium information!"

# Validate league statistics
invalid_results = gold_df.filter(
    col("played") != (
        col("won") + col("drawn") + col("lost")
    )
).count()

assert invalid_results == 0, \
    "Match statistics are inconsistent!"

print("Gold league table validation passed!")
print("Total teams:", total_rows)


In [0]:
# ==========================================
# 6. SAVE GOLD LEAGUE TABLE
# ==========================================

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_league_table")
)

print("Gold league table created successfully!")


In [0]:
%sql
SELECT
    position,
    team_name,
    stadium,
    played,
    won,
    drawn,
    lost,
    goal_difference,
    points,
    points_per_game,
    goals_per_game
FROM workspace.default.gold_league_table
ORDER BY position;
